# Solar Utility Agent — Training Notebook (Updated)
Downloads ~3 months of hourly Open-Meteo Archive weather, derives hourly GHI quantiles, labels hours directly, trains a 3-feature logistic regression, and saves `solar_model.pkl`.

In [ ]:
import joblib
import numpy as np
import pandas as pd
import requests
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report)

## 1. Config

In [ ]:
LAT, LON = 4.3596, 100.9761
START, END = "2024-01-01", "2024-03-31"   # ~3 months
FEATURES = ["hour", "ghi", "rain"]
CLASSES = ["GOOD", "MARGINAL", "NO"]

## 2. Pull Open-Meteo Archive & Derive Hourly Quantiles

In [ ]:
r = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": LAT, "longitude": LON,
        "start_date": START, "end_date": END,
        "hourly": "shortwave_radiation,precipitation",
        "timezone": "auto",
    }, timeout=60,
)
r.raise_for_status()
om = r.json()

df = pd.DataFrame({
    "time": pd.to_datetime(om["hourly"]["time"]),
    "ghi":  om["hourly"]["shortwave_radiation"],
    "rain": om["hourly"]["precipitation"],
})

# Scale precipitation (mm) to roughly match the 0-100% forecast scale
df["rain"] = df["rain"] * 10.0
df["hour"] = df["time"].dt.hour
df["date"] = df["time"].dt.date

# Compute quantiles from non-zero hourly GHI values only
non_zero_ghi = df[df["ghi"] > 0]["ghi"]
q33 = non_zero_ghi.quantile(0.33)
q66 = non_zero_ghi.quantile(0.66)
print(f"Hourly q33 = {q33:.3f} W/m², q66 = {q66:.3f} W/m²")

def assign_hourly_label(ghi):
    if ghi == 0:
        return "NO"
    if ghi >= q66:
        return "GOOD"
    if ghi >= q33:
        return "MARGINAL"
    return "NO"

df["label"] = df["ghi"].apply(assign_hourly_label)
print(df["label"].value_counts())

## 3. Train / Test Split (Jan–Feb train, Mar test)

In [ ]:
df["month"] = df["time"].dt.month
train = df[df["month"].isin([1, 2])]
test  = df[df["month"] == 3]

X_tr, y_tr = train[FEATURES].fillna(0), train["label"]
X_te, y_te = test[FEATURES].fillna(0),  test["label"]
print("train:", X_tr.shape, "test:", X_te.shape)

## 4. Fit Logistic Regression

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_tr, y_tr)
print("classes:", list(model.classes_))

## 5. Evaluation

In [ ]:
y_pred = model.predict(X_te)
print("accuracy :", round(accuracy_score(y_te, y_pred), 4))
print("macro-F1 :", round(f1_score(y_te, y_pred, average="macro"), 4))
print(confusion_matrix(y_te, y_pred, labels=CLASSES))
print(classification_report(y_te, y_pred, labels=CLASSES))

coef = pd.DataFrame(model.coef_, index=model.classes_, columns=FEATURES)
coef["intercept"] = model.intercept_
print(coef.round(3))

## 6. Learned FLOOR and GOOD Thresholds

In [ ]:
def utility_from_proba(p):
    d = dict(zip(model.classes_, p))
    return 1.0*d.get("GOOD", 0) + 0.5*d.get("MARGINAL", 0) + 0.0*d.get("NO", 0)

u_tr = np.array([utility_from_proba(p) for p in model.predict_proba(X_tr)])
FLOOR = float(u_tr[y_tr == "MARGINAL"].mean())
GOOD  = float(u_tr[y_tr == "GOOD"].mean())
print(f"FLOOR = {FLOOR:.3f}, GOOD = {GOOD:.3f}")

## 7. Save Artifact

In [ ]:
artifact = {
    "model":      model,
    "features":   FEATURES,
    "classes":    list(model.classes_),
    "floor":      FLOOR,
    "good":       GOOD,
    "q33":        float(q33),
    "q66":        float(q66),
    "trained_on": f"{START}..{END}",
    "location":   {"lat": LAT, "lon": LON},
}
joblib.dump(artifact, "solar_model.pkl")
print("saved updated solar_model.pkl")